In [2]:
import torch
from torch.utils.data import Dataset, DataLoader


# ---------------------------------------------------------
# 1. RAW TEXT
# ---------------------------------------------------------

text = "I am suraj pawar and I study until 4 am"


# ---------------------------------------------------------
# 2. SIMPLE WORD TOKENIZATION
# ---------------------------------------------------------

words = text.split()


# ---------------------------------------------------------
# 3. BUILD THE VOCABULARY
# ---------------------------------------------------------

vocab = []

for word in words:
    if word not in vocab:
        vocab.append(word)


# ---------------------------------------------------------
# 4. STRING-TO-INTEGER MAPPING
# ---------------------------------------------------------

stoi = {}

for index, word in enumerate(vocab):
    stoi[word] = index


# ---------------------------------------------------------
# 5. INTEGER-TO-STRING MAPPING
# ---------------------------------------------------------

itos = {}

for word, index in stoi.items():
    itos[index] = word


# ---------------------------------------------------------
# 6. ENCODE THE SENTENCE
# ---------------------------------------------------------

token_ids = []

for word in words:
    token_ids.append(stoi[word])


# ---------------------------------------------------------
# 7. CUSTOM NEXT-TOKEN DATASET
# ---------------------------------------------------------

class NextTokenDataset(Dataset):

    def __init__(self, token_ids, block_size):

        self.data = torch.tensor(
            token_ids,
            dtype=torch.long
        )

        self.block_size = block_size

    def __len__(self):

        number_of_examples = (
            len(self.data) - self.block_size
        )

        return number_of_examples

    def __getitem__(self, idx):

        start_x = idx
        end_x = idx + self.block_size

        start_y = idx + 1
        end_y = idx + self.block_size + 1

        x = self.data[start_x:end_x]
        y = self.data[start_y:end_y]

        return x, y


# ---------------------------------------------------------
# 8. CREATE THE DATASET
# ---------------------------------------------------------

dataset = NextTokenDataset(
    token_ids=token_ids,
    block_size=4
)


# ---------------------------------------------------------
# 9. CREATE THE DATALOADER
# ---------------------------------------------------------

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=False
)


# ---------------------------------------------------------
# 10. INSPECT EVERY BATCH
# ---------------------------------------------------------

for batch_number, (xb, yb) in enumerate(loader):

    print(f"\nBatch number: {batch_number}")

    print("Input token IDs:")
    print(xb)

    print("Target token IDs:")
    print(yb)

    print("Input shape:")
    print(xb.shape)

    print("Target shape:")
    print(yb.shape)

    print("Decoded examples:")

    for row_number in range(xb.shape[0]):

        input_words = []

        for token_id in xb[row_number]:
            integer_id = token_id.item()
            word = itos[integer_id]
            input_words.append(word)

        target_words = []

        for token_id in yb[row_number]:
            integer_id = token_id.item()
            word = itos[integer_id]
            target_words.append(word)

        print("Input :", input_words)
        print("Target:", target_words)


Batch number: 0
Input token IDs:
tensor([[0, 1, 2, 3],
        [1, 2, 3, 4]])
Target token IDs:
tensor([[1, 2, 3, 4],
        [2, 3, 4, 0]])
Input shape:
torch.Size([2, 4])
Target shape:
torch.Size([2, 4])
Decoded examples:
Input : ['I', 'am', 'suraj', 'pawar']
Target: ['am', 'suraj', 'pawar', 'and']
Input : ['am', 'suraj', 'pawar', 'and']
Target: ['suraj', 'pawar', 'and', 'I']

Batch number: 1
Input token IDs:
tensor([[2, 3, 4, 0],
        [3, 4, 0, 5]])
Target token IDs:
tensor([[3, 4, 0, 5],
        [4, 0, 5, 6]])
Input shape:
torch.Size([2, 4])
Target shape:
torch.Size([2, 4])
Decoded examples:
Input : ['suraj', 'pawar', 'and', 'I']
Target: ['pawar', 'and', 'I', 'study']
Input : ['pawar', 'and', 'I', 'study']
Target: ['and', 'I', 'study', 'until']

Batch number: 2
Input token IDs:
tensor([[4, 0, 5, 6],
        [0, 5, 6, 7]])
Target token IDs:
tensor([[0, 5, 6, 7],
        [5, 6, 7, 1]])
Input shape:
torch.Size([2, 4])
Target shape:
torch.Size([2, 4])
Decoded examples:
Input : ['an